# ARTI308 – Lab 6: Linear Regression on Ecommerce Customers

Applying the same Linear Regression model from the lab to the Ecommerce Customersdataset.  
Target variable: **Yearly Amount Spent**

## Step 1 – Load the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('Ecommerce_Customers')
print('Dataset loaded successfully.')
print('Shape:', df.shape)

## Step 2 – Explore the Data

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

**Observations:**
- 500 rows, 8 columns — no missing values.
- 3 non-numeric columns: Email, Address, Avatar (categorical/identifiers).
- 4 numeric features + 1 numeric target (Yearly Amount Spent).
- All numeric features are on similar scales (no extreme outliers).

## Step 3 – Data Cleaning

In [ ]:
# Check for nulls
print('Null values per column:')
print(df.isnull().sum())

# Drop non-informative text columns
df_clean = df.drop(columns=['Email', 'Address', 'Avatar'])
print('\nCleaned DataFrame shape:', df_clean.shape)

## Step 4 – Feature Engineering

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(df_clean.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nCorrelation with Yearly Amount Spent:')
print(df_clean.corr()['Yearly Amount Spent'].sort_values(ascending=False))

**Feature insights:**
- Length of Membership has the strongest correlation with the target (r = 0.81).
- Time on App is moderately correlated (r = 0.50).
- Time on Website has almost no linear relationship (r ≈ −0.003) — kept for completeness.
- No new features are needed.. all 4 numeric features are used as-is.

## Step 5 – Prepare the Data for Modeling

In [ ]:
X = df_clean.drop('Yearly Amount Spent', axis=1)
y = df_clean['Yearly Amount Spent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print('Training set size:', X_train.shape)
print('Test set size:    ', X_test.shape)

## Step 6 – Train the Model (Linear Regression)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)

print('Model Coefficients:')
print(coef_df.to_string(index=False))
print(f'\nIntercept: {model.intercept_:.4f}')

## Step 7 – Evaluate Model Performance

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('=== Model Evaluation ===')
print(f'MAE  : {mae:.4f}')
print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'R²   : {r2:.4f}')

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='steelblue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Yearly Amount Spent')
plt.ylabel('Predicted Yearly Amount Spent')
plt.title('Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals plot
residuals = y_test - y_pred
plt.figure(figsize=(7, 4))
plt.scatter(y_pred, residuals, alpha=0.5, color='coral')
plt.axhline(0, color='black', lw=1.5, linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted')
plt.tight_layout()
plt.show()

## Summary


**Key findings:**
- The Linear Regression model explains **98.1%** of the variance in yearly spending — an excellent fit.
- **Length of Membership** is by far the most influential feature (coef ≈ 61.7), meaning each additional year of membership is associated with ~\$62 more spent annually.
- **Time on App** matters significantly (coef ≈ 38.6), while **Time on Website** has almost no effect (coef ≈ 0.46).
- The residuals plot shows no clear pattern, confirming the linear model assumptions are well met.
- With an RMSE of ~\$10 on a target that averages ~\$499, predictions are very accurate.